In [1]:
import pandas as pd
import pyomo.environ as pyo
from food_security import optimizer

from pathlib import Path

In [2]:
src_dir = Path('~').expanduser() / 'OneDrive - Stichting Deltares/Documents - International Delta Toolset/Food security'

In [3]:
input_file = src_dir / 'input_all.xlsx'

crop_df = pd.read_excel(input_file, sheet_name='crop_id')
trade_df = pd.read_excel(input_file, sheet_name='trade')
production_df = pd.read_excel(input_file, sheet_name='production_d')
production_df = production_df[production_df['year'] == 2020]
price_df = pd.read_excel(input_file, sheet_name='market_price')
price_df = price_df.fillna(180)
theta_df = pd.read_excel(input_file, sheet_name='plant_based_diet')
energy_df = pd.read_excel(input_file, sheet_name='calories')
income_df = pd.read_excel(input_file, sheet_name='income')

population = 106540000

In [4]:
price_df = price_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["usd_per_tonnes"].mean()

production_df = production_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["yield"].sum()

trade_df = trade_df[trade_df['crop_name_fao'].isin(production_df['crop_name_fao'])]

energy_df = energy_df.groupby(
    ["crop_name_fao"],
    as_index=False
)["calories_kcal_kg"].mean()

input_df = (
    production_df
    .merge(price_df, on="crop_name_fao", how="left")
    .merge(trade_df, on="crop_name_fao", how="left")
    .merge(energy_df, on="crop_name_fao", how="left")
)

# Convert trade from tonnes to kg
input_df["import"] = input_df["import"] * 1000
input_df["export"] = input_df["export"] * 1000

input_df["quantity"] = input_df["yield"] + input_df["import"] - input_df["export"]

class_map = (
    crop_df
    .drop_duplicates(subset=["crop_name_fao"])
    .set_index("crop_name_fao")["class"]
)

input_df["class"] = input_df["crop_name_fao"].map(class_map)
input_df = input_df[input_df['class'] != '-']
input_df["quantity"] = input_df['quantity'].clip(lower=0, upper=None)
# Convert to daily availability
input_df["quantity"] = input_df["quantity"] / 365.25
input_df["usd_per_kg"] = input_df["usd_per_tonnes"] / 1000

In [5]:
FSOptimizer = optimizer.FSOptimizer(input_df=input_df, income_df=income_df, population=population, MDER=1806.61, lambda_g={
            g: 1.0 for g in ["carb", "protein", "fat", "fruitvege"]
        })
results = FSOptimizer.solve()

In [6]:
results[0]

,5,4,3,2,1
Barley,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
"Beans, dry",0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Cabbages,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Chick-peas,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Cucumbers and gherkins,3.904238e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Eggplants (aubergines),3.348491e+06,8.523200e+06,7.238474e+05,0.000000e+00,0.000000e+00
"Groundnuts, Shelled",2.885960e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Lemons and limes,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.604992e+06
Linseed,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Maize (incl. 0067 and 0068),0.000000e+00,0.000000e+00,0.000000e+00,1.449529e+06,1.044899e+07


In [7]:
results[1]

,5,4,3,2,1
Barley,0.000000,0.000000,0.000000,0.000000,0.000000
"Beans, dry",0.000000,0.000000,0.000000,0.000000,0.000000
Cabbages,0.000000,0.000000,0.000000,0.000000,0.000000
Chick-peas,0.000000,0.000000,0.000000,0.000000,0.000000
Cucumbers and gherkins,0.018323,0.000000,0.000000,0.000000,0.000000
Eggplants (aubergines),0.157147,0.400000,0.033971,0.000000,0.000000
"Groundnuts, Shelled",0.013544,0.000000,0.000000,0.000000,0.000000
Lemons and limes,0.000000,0.000000,0.000000,0.000000,0.356908
Linseed,0.000000,0.000000,0.000000,0.000000,0.000000
Maize (incl. 0067 and 0068),0.000000,0.000000,0.000000,0.068027,0.490379


In [8]:
results[2]

,5,4,3,2,1
objective function,4.398410e+06,7.446377e+06,7.625638e+06,7.700290e+06,8.835749e+06
spending_pp,2.064206e-01,3.494639e-01,3.578767e-01,3.613802e-01,4.146681e-01
consumed_energy_pp,1.806610e+03,1.806610e+03,1.806610e+03,1.806610e+03,1.806610e+03
carb_fraction,8.919190e-01,9.535041e-01,9.616083e-01,9.623604e-01,9.663116e-01
fat_fraction,5.866797e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
fruitvege_fraction,4.941303e-02,4.649592e-02,3.839170e-02,3.763956e-02,3.368842e-02
protein_fraction,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
carb_kg,1.827932e+00,7.110877e-01,7.171315e-01,6.857497e-01,4.903787e-01
fat_kg,1.951139e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
fruitvege_kg,4.000000e-01,4.000000e-01,4.000000e-01,4.000000e-01,4.000000e-01
